In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from glob import glob
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.cluster as sk
import plotly.graph_objects as go
from scipy.cluster.hierarchy import dendrogram
from scipy import stats

import thicket as tt

In [2]:
gpu_kernels = {
    "block_256": [
        "Algorithm_ATOMIC",
        "Algorithm_MEMCPY",
        "Algorithm_MEMSET",
        "Apps_DEL_DOT_VEC_2D",
        "Apps_EDGE3D",
        "Apps_ENERGY",
        "Apps_FIR",
        "Apps_LTIMES",
        "Apps_LTIMES_NOVIEW",
        "Apps_MATVEC_3D_STENCIL",
        "Apps_NODAL_ACCUMULATION_3D",
        "Apps_PRESSURE",
        "Apps_VOL3D",
        "Apps_ZONAL_ACCUMULATION_3D",
        "Basic_ARRAY_OF_PTRS",
        "Basic_COPY8",
        "Basic_DAXPY",
        "Basic_DAXPY_ATOMIC",
        "Basic_IF_QUAD",
        "Basic_INDEXLIST",
        "Basic_INDEXLIST_3LOOP",
        "Basic_INIT3",
        "Basic_INIT_VIEW1D",
        "Basic_INIT_VIEW1D_OFFSET",
        "Basic_MAT_MAT_SHARED",
        "Basic_MULADDSUB",
        "Basic_NESTED_INIT",
        "Basic_PI_ATOMIC",
        "Comm_HALO_EXCHANGE",
        "Comm_HALO_PACKING",
        "Comm_HALO_SENDRECV",
        "Lcals_DIFF_PREDICT",
        "Lcals_EOS",
        "Lcals_FIRST_DIFF",
        "Lcals_FIRST_SUM",
        "Lcals_GEN_LIN_RECUR",
        "Lcals_HYDRO_1D",
        "Lcals_HYDRO_2D",
        "Lcals_INT_PREDICT",
        "Lcals_PLANCKIAN",
        "Lcals_TRIDIAG_ELIM",
        "Polybench_2MM",
        "Polybench_3MM",
        "Polybench_ADI",
        "Polybench_ATAX",
        "Polybench_FDTD_2D",
        "Polybench_FLOYD_WARSHALL",
        "Polybench_GEMM",
        "Polybench_GEMVER",
        "Polybench_GESUMMV",
        "Polybench_HEAT_3D",
        "Polybench_JACOBI_1D",
        "Polybench_JACOBI_2D",
        "Polybench_MVT",
        "Stream_ADD",
        "Stream_COPY",
        "Stream_MUL",
        "Stream_TRIAD",
    ], # For block_256
    # "default": [
    #     "Algorithm_SORT",
    #     "Algorithm_SORTPAIRS",
    # ], # For default
#     "blkatm_occgs_256": [
#         "Algorithm_REDUCE_SUM",
#         "Basic_PI_REDUCE",
#         "Basic_REDUCE3_INT",
#         "Basic_REDUCE_STRUCT",
#         "Basic_TRAP_INT",
# #    ], "blkatm_direct_256": [
#         "Stream_DOT",
#     ], # For blkatm_occgs_256
    "block_64": [
        "Apps_CONVECTION3DPA",
        "Apps_DIFFUSION3DPA",
        "Apps_MASS3DEA",
    ], # For block_64
    "block_25": [
        "Apps_MASS3DPA",
    ], # For block_25
    # "funcptr_256": [
    #     "Comm_HALO_EXCHANGE_FUSED",
    #     "Comm_HALO_PACKING_FUSED",
    # ], # For funcptr_256
    # "rocprim": [
    #     "Algorithm_SCAN",
    # ], # For rocprim
#     "atomic_occgs_256": [
#         "Algorithm_HISTOGRAM",
# #    ], "atomic_direct_256": [
#         "Basic_MULTI_REDUCE",
#     ], # For atomic_occgs_256
    # "blkdev_occgs_256": [
    #     "Lcals_FIRST_MIN",
    # ], # For blkdev_occgs_256
    # "cub": [
    #     "Algorithm_SCAN",
    # ], # For cub
}

spr_kernels = {
    "default": [
        "Algorithm_ATOMIC",
        "Algorithm_HISTOGRAM",
        "Algorithm_MEMCPY",
        "Algorithm_MEMSET",
        "Algorithm_SCAN",
        "Algorithm_SORT",
        "Algorithm_SORTPAIRS",
        "Algorithm_REDUCE_SUM",
        "Apps_CONVECTION3DPA",
        "Apps_DEL_DOT_VEC_2D",
        "Apps_DIFFUSION3DPA",
        "Apps_EDGE3D",
        "Apps_ENERGY",
        "Apps_FIR",
        "Apps_LTIMES",
        "Apps_LTIMES_NOVIEW",
        "Apps_MASS3DEA",
        "Apps_MASS3DPA",
        "Apps_MATVEC_3D_STENCIL",
        "Apps_NODAL_ACCUMULATION_3D",
        "Apps_PRESSURE",
        "Apps_VOL3D",
        "Apps_ZONAL_ACCUMULATION_3D",
        "Basic_ARRAY_OF_PTRS",
        "Basic_COPY8",
        "Basic_DAXPY",
        "Basic_DAXPY_ATOMIC",
        "Basic_IF_QUAD",
        "Basic_INDEXLIST",
        "Basic_INDEXLIST_3LOOP",
        "Basic_INIT3",
        "Basic_INIT_VIEW1D",
        "Basic_INIT_VIEW1D_OFFSET",
        "Basic_MAT_MAT_SHARED",
        "Basic_MULADDSUB",
        "Basic_MULTI_REDUCE",
        "Basic_NESTED_INIT",
        "Basic_PI_ATOMIC",
        "Basic_PI_REDUCE",
        "Basic_REDUCE_STRUCT",
        "Basic_REDUCE3_INT",
        "Basic_TRAP_INT",
        "Comm_HALO_EXCHANGE",
        "Comm_HALO_PACKING",
        "Comm_HALO_SENDRECV",
        "Lcals_DIFF_PREDICT",
        "Lcals_EOS",
        "Lcals_FIRST_DIFF",
        "Lcals_FIRST_MIN",
        "Lcals_FIRST_SUM",
        "Lcals_GEN_LIN_RECUR",
        "Lcals_HYDRO_1D",
        "Lcals_HYDRO_2D",
        "Lcals_INT_PREDICT",
        "Lcals_PLANCKIAN",
        "Lcals_TRIDIAG_ELIM",
        "Polybench_2MM",
        "Polybench_3MM",
        "Polybench_ADI",
        "Polybench_ATAX",
        "Polybench_FDTD_2D",
        "Polybench_FLOYD_WARSHALL",
        "Polybench_GEMM",
        "Polybench_GEMVER",
        "Polybench_GESUMMV",
        "Polybench_HEAT_3D",
        "Polybench_JACOBI_1D",
        "Polybench_JACOBI_2D",
        "Polybench_MVT",
        "Stream_ADD",
        "Stream_COPY",
        "Stream_DOT",
        "Stream_MUL",
        "Stream_TRIAD",
    ], # default
    # "funcptr": [
    #     "Comm_HALO_EXCHANGE_FUSED",
    #     "Comm_HALO_PACKING_FUSED",
    # ], # For funcptr
}

# List of kernels to exclude from the data, agnostic of parameters
exclude_kernels = [
    "Algorithm_ATOMIC",
    "Algorithm_HISTOGRAM",
    "Algorithm_REDUCE_SUM",
    "Algorithm_SCAN",
    "Algorithm_SORT",
    "Algorithm_SORTPAIRS",

    "Apps_NODAL_ACCUMULATION_3D",

    "Basic_INDEXLIST",
    "Basic_INDEXLIST_3LOOP", 
    "Basic_MULTI_REDUCE",
    "Basic_PI_ATOMIC",
    "Basic_PI_REDUCE",
    "Basic_REDUCE_STRUCT",
    "Basic_REDUCE3_INT",
    "Basic_TRAP_INT",
    
    "Lcals_FIRST_MIN",

    "Comm_HALO_EXCHANGE",
    "Comm_HALO_EXCHANGE_FUSED",
    "Comm_HALO_SENDRECV",
    "Comm_HALO_EXCHANGE_FUSED",
    "Comm_HALO_PACKING_FUSED",

    "Polybench_GEMM",
    "Polybench_GEMVER",
    "Polybench_GESUMMV",

    "Stream_DOT",
]

In [3]:
Memory_bound = ['Algorithm_MEMCPY',
 'Algorithm_MEMSET',
 'Apps_ENERGY',
 'Apps_MATVEC_3D_STENCIL',
 'Apps_PRESSURE',
 'Apps_ZONAL_ACCUMULATION_3D',
 'Basic_ARRAY_OF_PTRS',
 'Basic_COPY8',
 'Basic_DAXPY',
 'Basic_DAXPY_ATOMIC',
 'Basic_IF_QUAD',
 'Basic_INIT3',
 'Basic_MULADDSUB',
 'Lcals_DIFF_PREDICT',
 'Lcals_EOS',
 'Lcals_FIRST_DIFF',
 'Lcals_FIRST_SUM',
 'Lcals_GEN_LIN_RECUR',
 'Lcals_HYDRO_1D',
 'Lcals_HYDRO_2D',
 'Lcals_INT_PREDICT',
 'Lcals_TRIDIAG_ELIM',
 'Polybench_ADI',
 'Polybench_FDTD_2D',
 'Polybench_JACOBI_1D',
 'Polybench_JACOBI_2D',
 'Stream_ADD',
 'Stream_COPY',
 'Stream_MUL',
 'Stream_TRIAD']

In [4]:
def kernel_query(kernel_list):
    return tt.query.Query().match(
        ".",
        lambda row: row["name"].apply(
            lambda n: n in kernel_list
        ).all()
    ).rel("*")

def not_kernel_query(kernel_list):
    return tt.query.Query().match(
        ".",
        lambda row: row["name"].apply(
            lambda n: n not in kernel_list
        ).all()
    ).rel("*")

In [5]:
#maindir = "/usr/workspace/thicket/"
maindir = "/Users/michaelmckinsey/Documents/data/"

In [6]:
data = maindir+"RAJAPerf-version-2024.07.0/grace-cascadelake-a100/notools_1GPU_results_GCC-11.3.0-CUDA-12.2.2/RAJA_CUDA/block_256/"

In [ ]:
tk = tt.Thicket.from_caliperreader(glob(data+"**/*.cali", recursive=True), fill_perfdata=False)

In [8]:
tk.metadata_columns_to_perfdata("jobsize")

In [9]:
# tk = tk.filter_metadata(lambda x: x["ProblemSizeRunParam"] > 600000)
# tk = tk.filter_metadata(lambda x: x["ProblemSizeRunParam"] < 20000000)

In [10]:
tk = tk.query(kernel_query(gpu_kernels["block_256"]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

In [11]:
tk = tk.query(kernel_query(Memory_bound), multi_index_mode="all")

In [12]:
tk2 = tk.deepcopy()

In [13]:
tk = tk.groupby("ProblemSizeRunParam").agg(np.mean)

In [14]:
timestep_mapping = {
    "Polybench_ADI": 4,
    "Polybench_FDTD_2D": 40,
    "Polybench_JACOBI_1D": 16,
    "Polybench_HEAT_3D": 20,
    "Polybench_JACOBI_2D": 40,
}

tk.dataframe["tsteps"] = tk.dataframe["name"].apply(lambda kernel_name: timestep_mapping[kernel_name] if kernel_name in timestep_mapping else 1)

In [15]:
tk.dataframe["BytesRead/Rep_mean"] = tk.dataframe["BytesRead/Rep_mean"] / tk.dataframe["tsteps"]
tk.dataframe["BytesWritten/Rep_mean"] = tk.dataframe["BytesWritten/Rep_mean"] / tk.dataframe["tsteps"]
tk.dataframe["Bytes/Rep_mean"] = tk.dataframe["Bytes/Rep_mean"] / tk.dataframe["tsteps"]

In [16]:
# MEM BW
tk.dataframe["Memory Bandwidth (GB/s)"] = (tk.dataframe["BytesRead/Rep_mean"] + tk.dataframe["BytesWritten/Rep_mean"]) * tk.dataframe["Reps_mean"] / tk.dataframe["Avg time/rank_mean"] / (10**9)

In [17]:
tk.dataframe["Bytes"] = (tk.dataframe["BytesRead/Rep_mean"] + tk.dataframe["BytesWritten/Rep_mean"]) * tk.dataframe["Reps_mean"]

In [18]:
tk.dataframe["Bytes/Rep/Problem Size"] = tk.dataframe["Bytes/Rep_mean"] / tk.dataframe["ProblemSize_mean"]

In [19]:
tk.dataframe["Avg time/rank/Kernel"] = tk.dataframe["Avg time/rank_mean"] / tk.dataframe["Reps_mean"] / tk.dataframe["Kernels/Rep_mean"]

In [20]:
tk.dataframe["Total ProblemSize"] = tk.dataframe["ProblemSize_mean"] * tk.dataframe["jobsize_mean"]

In [21]:
if tk.dataframe["jobsize_mean"].max() > 1:
    raise ValueError("These plots are not interpreted the same for multiple ranks")

In [22]:
pd.set_option('display.max_columns', 500)

In [23]:
plt.rcParams.update({
    'font.size': 12,})

In [ ]:
for node in set(tk.dataframe.index.get_level_values("node")):
    # Create a new figure and axis for each loop
    fig, ax = plt.subplots()

    # Add "Stream_TRIAD" data in red
    stream_triad_data = tk.dataframe.loc[tk.get_node("Stream_TRIAD")]
    sns.lineplot(
        data=stream_triad_data,
        x="Bytes/Rep_mean",
        y="Memory Bandwidth (GB/s)",
        marker="o",
        color='red',
        label="Stream_TRIAD",
        alpha=0.5,
        ax=ax  # Assigning the plot to the ax
    )
    
    sns.lineplot(
        data=tk.dataframe.loc[node], 
        x="Bytes/Rep_mean",
        y="Memory Bandwidth (GB/s)",
        marker="o",
        ax=ax  # Assigning the plot to the ax
    )

    ax.set_title(f'{node.frame["name"]} ({int(tk.dataframe.loc[node, "Kernels/Rep_mean"].iloc[0])} Kernels/rep)')

    l1 = 192000
    l1_sm = l1 * 108
    l2 = 40960000

    ax.axvline(x=l1_sm, color='orange', linestyle='--', label=f'L1 20MB')
    ax.axvline(x=l2, color='g', linestyle='--', label=f'L2 40MB')

    # Annotate each data point with the corresponding value
    for x, y, ps in zip(
        tk.dataframe.loc[node]["Bytes/Rep_mean"], 
        tk.dataframe.loc[node]["Memory Bandwidth (GB/s)"], 
        tk.dataframe.loc[node]["ProblemSize_mean"]
    ):
        if ps > 600000:
            ax.text(x, y-50, f"{ps/10**6:.0f}M", fontsize=8, va='top')
        else:
            ax.text(x, y-50, f"{ps/10**3:.0f}K", fontsize=8, va='top')

    for x, y, ps in zip(
        tk.dataframe.loc[tk.get_node("Stream_TRIAD")]["Bytes/Rep_mean"], 
        tk.dataframe.loc[tk.get_node("Stream_TRIAD")]["Memory Bandwidth (GB/s)"], 
        tk.dataframe.loc[tk.get_node("Stream_TRIAD")]["ProblemSize_mean"]
    ):
        if ps > 600000:
            ax.text(x, y+50, f"{ps/10**6:.0f}M", fontsize=8, va='bottom', alpha=0.5)
        else:
            ax.text(x, y+50, f"{ps/10**3:.0f}K", fontsize=8, va='bottom', alpha=0.5)

    ax.legend()
    ax.set_xscale('log', base=2)
    ax.set_xlim(tk.dataframe["Bytes/Rep_mean"].min(), tk.dataframe["Bytes/Rep_mean"].max())
    ax.set_ylim(tk.dataframe["Memory Bandwidth (GB/s)"].min(), tk.dataframe["Memory Bandwidth (GB/s)"].max())

    ax.set_xlabel("Bytes/Rep")

    # Show the plot for the current node
    plt.show()

    # Clear the figure after displaying it
    plt.clf()


In [25]:
# xaxis = "ProblemSize_mean"

In [26]:
# for node in set(tk.dataframe.index.get_level_values("node")):
#     fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
#     # Add "Stream_TRIAD" data in red
#     stream_triad_data = tk.dataframe.loc[tk.get_node("Stream_TRIAD")]
#     sns.lineplot(
#         data=stream_triad_data,
#         x=xaxis,
#         y="Memory Bandwidth (GB/s)",
#         marker="o",
#         ax=ax[0],
#         color='red',
#         label="Stream_TRIAD",
#         alpha=0.5,
#     )
#     sns.lineplot(
#         data=stream_triad_data,
#         x=xaxis,
#         y="Avg time/rank/Kernel",
#         marker="o",
#         ax=ax[1],
#         color='red',
#         label="Stream_TRIAD",
#         alpha=0.5,
#     )

#     # Node data
#     data = tk.dataframe.loc[node]
#     sns.lineplot(
#         data=data, 
#         x=xaxis,
#         y="Memory Bandwidth (GB/s)",
#         marker="o",
#         ax=ax[0],
#     )
#     sns.lineplot(
#         data=data, 
#         x=xaxis,
#         y="Avg time/rank/Kernel",
#         marker="o",
#         ax=ax[1],
#     )

#     print(data[["Memory Bandwidth (GB/s)", "Avg time/rank/Kernel"]])

#     l1 = 192000
#     l1_sm = l1 * 108
#     l2 = 40960000
#     HBM = 4e+10

#     # Normalize by bytes/problem to have ProblemSize on x-axis
#     # bytes_per_probsize = data["Bytes/Rep/Problem Size"].iloc[0]
#     # l1 = l1 / bytes_per_probsize
#     # l1_sm = l1_sm / bytes_per_probsize
#     # l2 = l2 / bytes_per_probsize
#     # l3 = l3 / bytes_per_probsize

#     # Apply log scale to both subplots
#     ax[0].set_xscale('log', base=2)
#     ax[1].set_xscale('log', base=2)

#     def format_large_numbers(value):
#         if value >= 1_000_000:
#             return f"{value / 1_000_000:.0f}M"
#         elif value >= 1_000:
#             return f"{value / 1_000:.0f}K"
#         else:
#             return str(value)

#     # Set formatted x-ticks for both subplots
#     for i in range(2):
#         ax[i].set_xticks(data[xaxis])
#         ax[i].set_xticklabels([format_large_numbers(x) for x in data[xaxis]], rotation=45)

#     # Add second x-axis (Bytes/Rep_mean)
#     ax2_0 = ax[0].twiny()
#     ax2_1 = ax[1].twiny()

#     ax2_0.axvline(x=l1_sm, color='orange', linestyle='--', label='L1_SM 20MB') # -> (192KB * 108SM = 20MB)
#     ax2_0.axvline(x=l2, color='g', linestyle='--', label=f'L2 40MB')
#     # ax2_0.axvline(x=l3, color='r', linestyle='--', label=f'L3 262MB')
#     ax2_1.axvline(x=l1_sm, color='orange', linestyle='--', label='L1_SM 20MB') # -> (192KB * 108SM = 20MB)
#     ax2_1.axvline(x=l2, color='g', linestyle='--', label=f'L2 40MB')
#     ax2_0.legend()

#     ax2_0.set_xscale('log', base=2)
#     ax2_1.set_xscale('log', base=2)
    
#     byte_ticks =  data["Bytes/Rep_mean"]# * data["Bytes/Rep/Problem Size"]#data[xaxis].to_numpy()
#     ax2_0.set_xticks(byte_ticks)
#     ax2_1.set_xticks(byte_ticks)
    
#     # Assuming 'Bytes/Rep_mean' exists in the dataframe:
#     byte_labels = [int(x) for x in list(byte_ticks)]
#     byte_labels = [format_large_numbers(x) for x in byte_labels]
    
#     ax2_0.set_xticklabels(byte_labels, rotation=45)
#     ax2_1.set_xticklabels(byte_labels, rotation=45)
    
#     ax2_0.set_xlabel("Bytes/Rep")
#     ax2_1.set_xlabel("Bytes/Rep")

#     # Sync limits and add custom ticks/labels for Bytes/Rep_mean
#     # ax2_0.set_xlim(ax[0].get_xlim())
#     # ax2_1.set_xlim(ax[1].get_xlim())

#     # strip _mean
#     for i in range(2):
#         current_label = ax[i].get_xlabel()
#         ax[i].set_xlabel(current_label.replace("_mean", " per GPU"))

#     fig.suptitle(f'{node.frame["name"]} ({int(tk.dataframe.loc[node, "Kernels/Rep_mean"].iloc[0])} Kernels/rep)')
#     fig.tight_layout()

#     plt.show()


In [27]:
# for node in set(tk.dataframe.index.get_level_values("node")):
#     fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
#     # Add "Stream_TRIAD" data in red
#     stream_triad_data = tk.dataframe.loc[tk.get_node("Stream_TRIAD")]
#     sns.lineplot(
#         data=stream_triad_data,
#         x="Total ProblemSize",
#         y="Memory Bandwidth (GB/s)",
#         marker="o",
#         ax=ax[0],
#         color='red',
#         label="Stream_TRIAD",
#         alpha=0.5,
#     )
#     sns.lineplot(
#         data=stream_triad_data,
#         x="Total ProblemSize",
#         y="Avg time/rank/Kernel",
#         marker="o",
#         ax=ax[1],
#         color='red',
#         label="Stream_TRIAD",
#         alpha=0.5,
#     )

#     # Node data
#     data = tk.dataframe.loc[node]
#     sns.lineplot(
#         data=data, 
#         x="Total ProblemSize",
#         y="Memory Bandwidth (GB/s)",
#         marker="o",
#         ax=ax[0],
#     )
#     sns.lineplot(
#         data=data, 
#         x="Total ProblemSize",
#         y="Avg time/rank/Kernel",
#         marker="o",
#         ax=ax[1],
#     )

#     print(data[["Memory Bandwidth (GB/s)", "Avg time/rank/Kernel"]])

#     fig.suptitle(f'{node.frame["name"]} ({int(tk.dataframe.loc[node, "Kernels/Rep_mean"].iloc[0])} Kernels/rep)')

#     l1 = 192000
#     l1_sm = l1 * 108
#     l2 = 40960000
#     HBM = 4e+10

#     # Normalize by bytes/problem to have ProblemSize on x-axis
#     bytes_per_probsize = data["Bytes/Rep/Problem Size"].iloc[0]
#     l1_sm = l1_sm / bytes_per_probsize
#     l2 = l2 / bytes_per_probsize
#     HBM = HBM / bytes_per_probsize

#     ax[0].axvline(x=l1_sm, color='orange', linestyle='--', label='L1_SM 20MB') # -> (192KB * 108SM = 20MB)
#     ax[0].axvline(x=l2, color='g', linestyle='--', label=f'L2 40MB')
#     #ax[0].axvline(x=HBM, color='r', linestyle='--', label=f'HBM 40GB')
#     ax[1].axvline(x=l1_sm, color='orange', linestyle='--', label='L1_SM 20MB') # -> (192KB * 108SM = 20MB)
#     ax[1].axvline(x=l2, color='g', linestyle='--', label=f'L2 40MB')
#     ax[0].legend()

#     # Apply log scale to both subplots
#     ax[0].set_xscale('log', base=2)
#     ax[1].set_xscale('log', base=2)

#     def format_large_numbers(value):
#         if value >= 1_000_000:
#             return f"{value / 1_000_000:.0f}M"
#         elif value >= 1_000:
#             return f"{value / 1_000:.0f}K"
#         else:
#             return str(value)

#     # Set formatted x-ticks for both subplots
#     for i in range(2):
#         ax[i].set_xticks(data["Total ProblemSize"])
#         ax[i].set_xticklabels([format_large_numbers(x) for x in data["Total ProblemSize"]], rotation=45)

#     # strip _mean
#     for i in range(2):
#         current_label = ax[i].get_xlabel()
#         ax[i].set_xlabel(current_label.replace("_mean", ""))


#     plt.show()


In [28]:
# for node in set(tk.dataframe.index.get_level_values("node")):
#     sns.lineplot(
#         data=tk.dataframe.loc[node], 
#         x="Bytes/Rep_mean",
#         y="Memory Bandwidth (GB/s)",
#         marker="o"   
#     )
#     plt.title(f'{node.frame["name"]} ({int(tk.dataframe.loc[node, "Kernels/Rep_mean"].iloc[0])} Kernels/rep)')

#     l1 = 192000
#     l1_sm = l1 * 108
#     l2 = 40960000


#     plt.axvline(x=l1_sm, color='orange', linestyle='--', label=f'L1_SM = {l1_sm} (192KB * 108SM = 20MB)')
#     plt.axvline(x=l2, color='g', linestyle='--', label=f'L2 = {l2} (40MB)')

#     # Annotate each data point with the corresponding value
#     for x, y, ps in zip(
#         tk.dataframe.loc[node]["Bytes/Rep_mean"], 
#         tk.dataframe.loc[node]["Memory Bandwidth (GB/s)"], 
#         tk.dataframe.loc[node]["ProblemSize_mean"]
#     ):
#         plt.text(x, y, f"{ps/10**3:.0f}K", fontsize=9, ha='right', va='bottom')

#     plt.legend()
#     plt.xscale('log', base=2)

#     plt.show()
